In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import unicodedata


sns.set_theme(style="whitegrid")
df = pd.read_json("jobs.jl", lines=True)

In [ ]:
df.info()
df.head()

In [ ]:
def categorize_experience(x):
    if pd.isna(x):
        return 'Unknown'
    x = str(x).lower()
    if 'junior' in x or 'jr' in x:
        return 'Junior'
    elif 'middle' in x or 'mid' in x:
        return 'Mid'
    elif 'senior' in x or 'sr' in x:
        return 'Senior'
    else:
        return 'Other'

df['level'] = df['experience'].apply(categorize_experience)
df['level'].value_counts()

In [ ]:
titles = ' '.join(df['title'].dropna())

titles = unicodedata.normalize("NFKD", titles).encode("ascii", "ignore").decode()
titles = re.sub(r"[^a-zA-Z\s]", " ", titles).lower()

words = [w for w in titles.split() if len(w) > 2]

stopwords = {
    "and","for","with","the","job","engineer","developer","senior","junior",
    "lead","middle","mid","level","remote","full","stack","team","tech",
    "software","digital","manager","specialist","analyst","product","project"
}

words = [w for w in words if w not in stopwords]

common_words = Counter(words).most_common(20)

pd.DataFrame(common_words, columns=["word", "count"]).plot.bar(
    x="word", y="count", figsize=(12,5)
)

plt.title("Top Words in Job Titles")
plt.show()

In [ ]:
title_levels = df.dropna(subset=['title']).groupby(['level','title']).size().unstack(fill_value=0)

plt.figure(figsize=(14,6))
sns.heatmap(title_levels, cmap='YlGnBu')
plt.title("Job Titles vs Experience Level")
plt.xlabel("Job Title")
plt.ylabel("Experience Level")
plt.show()

In [ ]:
df_to_save = df[['title', 'level']]
df_to_save.to_csv("jobs_analysis.csv", index=False)
print("Processed data saved to jobs_analysis.csv")